# Apprentissage Federe pour la Confidentialite des Donnees

## Contexte et Objectifs

Ce notebook presente une implementation pratique de l'Apprentissage Federe (Federated Learning), une approche de l'apprentissage automatique qui permet d'entrainer des modeles sur des donnees distribuees sans que ces donnees ne quittent jamais l'appareil local. C'est une technique fondamentale pour construire des systemes d'IA respectueux de la vie privee.

Nous simulons un scenario d'apprentissage federe en utilisant le jeu de donnees MNIST, ou les donnees d'entrainement sont reparties entre plusieurs "clients" virtuels.

### Concepts Cles de ce Notebook :

1.  **Simulation d'un Environnement Federe :** Nous divisons le jeu de donnees MNIST pour simuler un reseau de 10 clients, chacun possedant une partie des donnees.
2.  **Modele Local et Global :** Nous definissons un modele de reseau de neurones simple (CNN) qui sera entraine localement sur chaque client. Le serveur central maintiendra un modele global.
3.  **Algorithme Federated Averaging (FedAvg) :** Nous implementons l'algorithme FedAvg, qui est le pilier de l'apprentissage federe :
    *   **Mise a jour du Client (Client Update) :** Chaque client entraine le modele sur ses propres donnees locales.
    *   **Agregation du Serveur (Server Aggregation) :** Le serveur central recupere les poids des modeles mis a jour par les clients et les moyenne pour creer un nouveau modele global ameliore.
4.  **Boucle d'Entrainement Federee :** La simulation se deroule sur plusieurs "rounds" de communication, montrant comment le modele global s'ameliore sans jamais voir les donnees brutes.
5.  **Evaluation de la Performance :** Nous evaluons l'exactitude du modele global sur un jeu de test centralise pour mesurer l'efficacite de l'apprentissage.

_Derniere mise a jour : 2026-02-16_

In [1]:
# --- 1. Installation des Dependances ---
%pip install -q torch torchvision numpy matplotlib
print("Dependances installees.")

SyntaxError: invalid syntax (2051176013.py, line 1)

In [2]:
# --- 2. Imports et Configuration ---
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset
import numpy as np
import copy
import matplotlib.pyplot as plt
import logging

# --- Configuration ---
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

SyntaxError: invalid syntax (3714364521.py, line 1)

## 3. Preparation des Donnees et Simulation de l'Environnement Federe

Nous chargeons MNIST et le divisons en sous-ensembles pour simuler 10 clients distincts, chacun avec sa propre partition de donnees.

In [3]:
# --- Hyperparametres ---
num_clients = 10
batch_size = 64
epochs_per_client = 5
learning_rate = 0.01
num_rounds = 5 # Nombre de cycles de communication federee

# --- Chargement et Partitionnement des Donnees ---
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Diviser le jeu d'entrainement pour les clients
num_data_per_client = len(train_dataset) // num_clients
lengths = [num_data_per_client] * (num_clients - 1)
lengths.append(len(train_dataset) - sum(lengths)) # Assurer que toutes les donnees sont utilisees
client_datasets = random_split(train_dataset, lengths)

# Creer un DataLoader pour chaque client
client_loaders = [DataLoader(subset, batch_size=batch_size, shuffle=True) for subset in client_datasets]
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

logger.info(f"Donnees reparties entre {num_clients} clients, chacun avec environ {num_data_per_client} images.")

SyntaxError: invalid syntax (823652641.py, line 1)

## 4. Definition du Modele (CNN)

Un reseau de neurones a convolution simple, adapte pour la classification d'images comme MNIST.

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(1024, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(nn.functional.relu(self.conv1(x)))
        x = self.pool(nn.functional.relu(self.conv2(x)))
        x = x.view(-1, 1024)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return nn.functional.log_softmax(x, dim=1)

# Initialiser le modele global
global_model = SimpleCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
global_model.to(device)

SyntaxError: invalid syntax (3041504807.py, line 1)

## 5. Implementation de l'Apprentissage Federe (FedAvg)

In [5]:
def client_update(client_loader, model):
    """Entraine le modele sur les donnees d'un client."""
    model.train()
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    criterion = nn.NLLLoss()
    
    for _ in range(epochs_per_client):
        for data, target in client_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
    return model.state_dict()

def server_aggregate(global_model, client_models):
    """Agrege les poids des clients pour mettre a jour le modele global."""
    global_dict = global_model.state_dict()
    
    # Moyenne des poids
    for k in global_dict.keys():
        global_dict[k] = torch.stack([client_models[i][k].float() for i in range(len(client_models))], 0).mean(0)
    
    global_model.load_state_dict(global_dict)
    return global_model

def evaluate(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    criterion = nn.NLLLoss()
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    return test_loss, accuracy

SyntaxError: invalid syntax (416476538.py, line 1)

## 6. Boucle d'Entrainement Federee et Resultats

In [ ]:
# --- Boucle Principale ---
accuracy_history = []
logger.info("Debut de l'entrainement federe...")

for round_num in range(1, num_rounds + 1):
    # Selectionner un sous-ensemble de clients (ici, tous les clients pour la simplicite)
    selected_clients = range(num_clients)
    
    client_models = []
    # Entrainement local sur les clients selectionnes
    for client_idx in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model_state = client_update(client_loaders[client_idx], local_model)
        client_models.append(local_model_state)
    
    # Agregation sur le serveur
    global_model = server_aggregate(global_model, client_models)
    
    # Evaluation du modele global
    loss, accuracy = evaluate(global_model, test_loader)
    accuracy_history.append(accuracy)
    logger.info(f'Round {round_num}/{num_rounds} | Perte: {loss:.4f} | Exactitude: {accuracy:.2f}%')

logger.info("Entrainement federe termine.")

# --- Visualisation de la Progression ---
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_rounds + 1), accuracy_history, marker='o')
plt.title("Evolution de l'Exactitude du Modele Global")
plt.xlabel("Round de Communication")
plt.ylabel("Exactitude sur le Test (%)")
plt.xticks(range(1, num_rounds + 1))
plt.grid(True)
plt.show()